# CP2 · Examen blanc de la partie II — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/checkpoints/partie_2/03_examen_solutions.ipynb)

L'enregistrement des réponses que vérifie le notebook de l'examen (cellules marquées `answer`, récoltées par `tools/build_answers.py`), calculées à partir des nombres du sujet ; les erreurs classiques reçoivent un message qui dit où chercher. Le corrigé détaillé et le barème sont dans `03_examen_corrige.md`.

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast

🧪 workbookMaths : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.18 | numpy 2.1.3 | scipy 1.16.3 | sympy 1.14.0 | matplotlib 3.10.0 | pandas 2.2.3 | scikit-learn 1.6.1
   Graine : 42 | FAST_MODE : True


## Les réponses vérifiées après l'examen

In [2]:
# The answers are computed from the numbers of the statements (01_examen_sujet.md)
import math
from math import comb


def dot(a, b):
    return sum(x * y for x, y in zip(a, b))


def solve2(m, v):
    """The solution of the 2 x 2 system m w = v (Cramer)."""
    (a, b), (c, d) = m
    det = a * d - b * c
    return [(d * v[0] - b * v[1]) / det, (a * v[1] - c * v[0]) / det]


# CP2.1: u, v, the hyperplane w · x + b = 0 and the point P
U_1, V_1 = (2, -1, 2, 4), (3, 1, 0, 2)
W_1, B_1, P_1 = (1, -2, 2, 4), -7, (2, 0, 1, 3)
PROJ_1 = [dot(U_1, V_1) / dot(U_1, U_1) * x for x in U_1]
S_1 = dot(W_1, P_1) + B_1

# CP2.2: 12 winning games out of 15 under H0: p = 1/2, then the Wilson and Wald intervals
N_2, K_2, Z = 15, 12, 1.96
TAIL_2 = sum(comb(N_2, j) for j in range(K_2, N_2 + 1)) / 2 ** N_2
PH_2 = K_2 / N_2
CENTER_2 = (PH_2 + Z ** 2 / (2 * N_2)) / (1 + Z ** 2 / N_2)
HALF_2 = Z / (1 + Z ** 2 / N_2) * math.sqrt(PH_2 * (1 - PH_2) / N_2 + Z ** 2 / (4 * N_2 ** 2))
WALD_HIGH_2 = PH_2 + Z * math.sqrt(PH_2 * (1 - PH_2) / N_2)

# CP2.3: twelve p-values, then three models (k, ln L) on n = 80 examples
P_3 = [0.0008, 0.0039, 0.011, 0.018, 0.027, 0.044, 0.07, 0.13, 0.21, 0.38, 0.55, 0.83]
MODELS_3, N_3 = [(2, -152.3), (4, -148.1), (7, -145.9)], 80
AIC_3 = [2 * k - 2 * ll for k, ll in MODELS_3]
BIC_3 = [k * math.log(N_3) - 2 * ll for k, ll in MODELS_3]

# CP2.5: X (4 x 2), y, least squares and ridge with lambda = 3
X_5 = [(1, 0), (1, 1), (0, 2), (2, 1)]
Y_5 = [2, 3, 4, 3]
G_5 = [[sum(r[i] * r[j] for r in X_5) for j in range(2)] for i in range(2)]
XY_5 = [sum(r[i] * y for r, y in zip(X_5, Y_5)) for i in range(2)]
W_5 = solve2(G_5, XY_5)
R_5 = [y - dot(r, W_5) for r, y in zip(X_5, Y_5)]
WR_5 = solve2([[G_5[0][0] + 3, G_5[0][1]], [G_5[1][0], G_5[1][1] + 3]], XY_5)
SCR_RIDGE_5 = sum((y - dot(r, WR_5)) ** 2 for r, y in zip(X_5, Y_5))


# CP2.6: p̃ = (K + 1)/(n + 2) for n = 8 Bernoulli(p) trials
def laplace_6(p, n=8):
    """Bias, variance and mean squared error of (K + 1)/(n + 2)."""
    bias = (n * p + 1) / (n + 2) - p
    variance = n * p * (1 - p) / (n + 2) ** 2
    return bias, variance, variance + bias ** 2


# CP2.7: the margin of a separator for the two labelled clouds
POS_7, NEG_7 = [(4, 4), (5, 6), (6, 4)], [(1, 1), (2, 3), (0, 3)]


def margin_7(w, b):
    scores = [dot(w, x) + b for x in POS_7] + [-(dot(w, x) + b) for x in NEG_7]
    return min(scores) / math.hypot(*w)


# CP2.11: six nights, the line by the normal equations, then three AIC
X_11, Y_11 = [1, 2, 4, 5, 7, 11], [7, 9, 14, 15, 18, 27]
N_11 = len(X_11)
G_11 = [[N_11, sum(X_11)], [sum(X_11), sum(x * x for x in X_11)]]
XY_11 = [sum(Y_11), sum(x * y for x, y in zip(X_11, Y_11))]
A_11, B_11 = solve2(G_11, XY_11)
SCR_11 = sum((y - A_11 - B_11 * x) ** 2 for x, y in zip(X_11, Y_11))
SCT_11 = sum((y - sum(Y_11) / N_11) ** 2 for y in Y_11)


def aic_11(scr, k):
    return N_11 * math.log(scr / N_11) + 2 * k


LOW_11 = 344 / 400 - math.sqrt(math.log(1 / 0.05) / (2 * 400))

**CP2.1 — Normes, cosinus, projection et distance à un hyperplan**

In [3]:
wbm.record("CP2.1a", [9, 5, 4], mistakes={"‖u‖₂ est la racine de la somme des carrés": [9, 25, 4], "‖u‖₁ additionne les valeurs absolues des coordonnées": [7, 5, 4]})
wbm.record("CP2.1b", dot(U_1, V_1), mistakes={"attention au signe de la deuxième coordonnée de u": 15})
wbm.record("CP2.1c", dot(U_1, V_1) / math.sqrt(dot(U_1, U_1) * dot(V_1, V_1)), decimals=3, mistakes={"le cosinus divise le produit scalaire par le produit des deux normes": {13 / 25, 13 / 14}, "les normes sont des racines carrées : ‖v‖ n'est pas la somme des carrés": 13 / 70})
wbm.record("CP2.1d", PROJ_1, decimals=2, mistakes={"projeter sur la droite dirigée par u divise u · v par ‖u‖², pas par ‖u‖": [13 / 5 * x for x in U_1], "c'est la projection de u sur v ; la question projette v sur u": [13 / 14 * x for x in V_1], "c'est la composante orthogonale ; la question demande la projection": [v - p for v, p in zip(V_1, PROJ_1)]})
wbm.record("CP2.1e", math.dist(V_1, PROJ_1), decimals=3, mistakes={"c'est le carré de la distance": dot(V_1, V_1) - 169 / 25, "c'est la norme de la projection ; la distance est celle de v à sa projection": 13 / 5})
wbm.record("CP2.1f", S_1 / math.sqrt(dot(W_1, W_1)), decimals=2, mistakes={"la distance signée divise w · P + b par ‖w‖, pas par ‖w‖²": S_1 / 25, "divise w · P + b par la norme de w": S_1, "l'équation s'écrit w · x + b = 0, avec b = −7": 23 / 5})
wbm.record("CP2.1g", [p - S_1 / dot(W_1, W_1) * w for p, w in zip(P_1, W_1)], decimals=2, mistakes={"P est du côté où w · x + b > 0 : on recule dans le sens de −w": [p + S_1 / 25 * w for p, w in zip(P_1, W_1)], "le coefficient de w est (w · P + b) / ‖w‖², pour arriver sur H": [p - S_1 / 5 * w for p, w in zip(P_1, W_1)]})

WBM_ANSWER {"decimals": 0, "element_hashes": ["216fd405a4fd05b08453594476a6aefca01217756549828b5b29aad190e5187e", "3cb9747226596f0ee635a4f0166f3320cfc998cccd7341ca45f3b16331f149ee", "4da62392b7242c810b6881f98cbf9cb5be4e214973727490daea691697f9bcd5"], "hash": "d745bd4b544e29b6f55901e1d2850e56f30bdd651a6cbb68f7edc0346dff9fb1", "id": "CP2.1a", "integer": true, "kind": "array", "mistakes": {"80a8355522f14857564701c1cf969c8f80cc5f69fd57f84cb189d87b9fbd63e2": "‖u‖₁ additionne les valeurs absolues des coordonnées", "c5926025b8659f3eba975fa506ceab4b8e0235746dbd35ccbd16918d817e6ded": "‖u‖₂ est la racine de la somme des carrés"}, "shape": [3]}
📝 Ex CP2.1a : réponse enregistrée (array, 0 décimale).
WBM_ANSWER {"hash": "44277aca63d3ff3ed1249042dddb13bcb8a06f1b640c682bcaa172f6664c07ae", "id": "CP2.1b", "kind": "int", "magnitude": 1, "mistakes": {"ebabd8d1b39fcb54a5ce951834651784265a9167239a6dcafd94e963eb8f41ab": "attention au signe de la deuxième coordonnée de u"}, "sign": 1}
📝 Ex CP2.1b : réponse 

**CP2.2 — Un test binomial et un intervalle de Wilson**

In [4]:
wbm.record("CP2.2a", comb(15, 12) / 2 ** 15, decimals=4, mistakes={"compte toutes les façons de placer les 12 parties gagnantes : le coefficient binomial": 1 / 2 ** 15})
wbm.record("CP2.2b", TAIL_2, decimals=4, mistakes={"« au moins 12 » compte aussi 13, 14 et 15 parties gagnantes": comb(15, 12) / 2 ** 15, "P(K ≥ 12) contient le cas K = 12": (comb(15, 13) + comb(15, 14) + 1) / 2 ** 15})
wbm.record("CP2.2c", 2 * TAIL_2, decimals=4, mistakes={"la loi de K est symétrique sous H0 : le test bilatéral compte les deux côtés": TAIL_2 / 2})
wbm.record("CP2.2d", True, mistakes={"compare la p-valeur bilatérale au seuil de 0,05": False})
wbm.record("CP2.2e", 12, mistakes={"calcule P(K ≥ 11) sous H0 et compare-la à 0,05": 11, "P(K ≥ 12) est-elle déjà sous 0,05 ?": 13})
wbm.record("CP2.2f", [CENTER_2 - HALF_2, CENTER_2 + HALF_2], decimals=4, mistakes={"c'est l'intervalle de Wald ; Wilson décale le centre vers 1/2 et change la demi-largeur": [PH_2 - Z * math.sqrt(PH_2 * (1 - PH_2) / N_2), WALD_HIGH_2]})
wbm.record("CP2.2g", WALD_HIGH_2, decimals=4, mistakes={"c'est la borne haute de Wilson ; la question demande celle de Wald": CENTER_2 + HALF_2, "l'erreur type de Wald divise p̂ (1 − p̂) par n": PH_2 + Z * math.sqrt(PH_2 * (1 - PH_2) / (N_2 - 1))})

WBM_ANSWER {"coarse_any": {"2": ["40e227084e55931c1e519f2d7c6431ec37e04efb46bdbeef35ba0cecab070136"]}, "decimals": 4, "hash": "3fa81f2a341fa75b90962da3a18673fea98eb05d0517f71ffba2d7a4f7be0546", "hash_coarse": "a639a53cd0d4bf39946c93687fd0be92d1ce9bf7ca79cdfce9dfdd6ae25b70a5", "id": "CP2.2a", "kind": "float", "magnitude": -2, "mistakes": {"dcf3d235facc5ea9932ea8c32f541c76b61dd4f73ed8a9b9096d51eba4c3e06f": "compte toutes les façons de placer les 12 parties gagnantes : le coefficient binomial"}, "sign": 1}
📝 Ex CP2.2a : réponse enregistrée (float, 4 décimales).
WBM_ANSWER {"coarse_any": {"2": ["b29bb41f86ca1332c966993ec740c4fda252843e0819960546e8084aa2bfffe6"]}, "decimals": 4, "hash": "75f3b4e13763b87b85878d3083f955754d52f8795c20baf6559c58afef2a255a", "hash_coarse": "6c84c123066fbc0f8a6dde49c307ba4ab4815ed110409b6fa7caabb3386510dd", "id": "CP2.2b", "kind": "float", "magnitude": -2, "mistakes": {"77dbb410f5ea718f55e0aa77bf18ff6698188cac6962c00ed67e90c992e63a5a": "« au moins 12 » compte aus

**CP2.3 — Comparaisons multiples, AIC et BIC**

In [5]:
wbm.record("CP2.3a", sum(p <= 0.05 for p in P_3), mistakes={"« significatif au seuil de 5 % » veut dire p ≤ 0,05 : compte-les toutes": 5})
wbm.record("CP2.3b", sum(p <= 0.05 / 12 for p in P_3), mistakes={"Bonferroni compare chaque p-valeur au seuil divisé par le nombre de tests": {1, 6}})
wbm.record("CP2.3c", min(1, 12 * P_3[2]), decimals=3, mistakes={"la p-valeur ajustée de Bonferroni multiplie la p-valeur (sans dépasser 1)": P_3[2] / 12, "Bonferroni multiplie par le nombre total de tests, pas par le rang": 3 * P_3[2]})
wbm.record("CP2.3d", AIC_3, decimals=1, mistakes={"AIC = 2k − 2 ln L : la log-vraisemblance est multipliée par −2": [2 * k - ll for k, ll in MODELS_3], "la pénalité de l'AIC est 2k": [k - 2 * ll for k, ll in MODELS_3]})
wbm.record("CP2.3e", BIC_3, decimals=1, mistakes={"le BIC utilise le logarithme népérien de n": [k * math.log10(N_3) - 2 * ll for k, ll in MODELS_3], "la pénalité du BIC est k ln n": AIC_3})
wbm.record("CP2.3f", [2, 1], mistakes={"on retient le modèle de plus petit AIC, et de plus petit BIC": [1, 3], "le BIC pénalise chaque paramètre plus fort que l'AIC : compare tes trois BIC": [2, 2]})

WBM_ANSWER {"hash": "2c284c4d39acda8ddb5c64324398e87d87ef5cdc1924b6a8cca866a5d9f38c59", "id": "CP2.3a", "kind": "int", "magnitude": 0, "mistakes": {"658250d62b2d90ff2a26db7af5c9415e4078ba5bc92c473d7179b6922058fc70": "« significatif au seuil de 5 % » veut dire p ≤ 0,05 : compte-les toutes"}, "sign": 1}
📝 Ex CP2.3a : réponse enregistrée (int).
WBM_ANSWER {"hash": "00d9cc48a43607055cf334b9645b0e9de931c56b1a7fda9606aae1d1a37f77a2", "id": "CP2.3b", "kind": "int", "magnitude": 0, "mistakes": {"1b49e6a48bce0a5bab5585dfa22e4ba19b0277d8331297bb62c0bad8e7937278": "Bonferroni compare chaque p-valeur au seuil divisé par le nombre de tests", "d5b14a8f5ae3381ffe9de30fb58fc1430be20d281830c10a58821f261149c122": "Bonferroni compare chaque p-valeur au seuil divisé par le nombre de tests"}, "sign": 1}
📝 Ex CP2.3b : réponse enregistrée (int).
WBM_ANSWER {"coarse_any": {"1": ["59f13fa48b81e1a632763979b3c9ab88feb71b5bdf48954bc034d45eb0ed8fde"]}, "decimals": 3, "hash": "1ee61535c0cac8469adc503eccbaa66a61b7b5

**CP2.4 — Pivot de Gauss, rang et inverse**

In [6]:
wbm.record("CP2.4a", [0, 3, 3, 12], mistakes={"L₃ ← L₃ + L₁ : on ajoute la première ligne": [-2, -1, 5, 6], "l'opération porte aussi sur le second membre": [0, 3, 3, 9]})
wbm.record("CP2.4b", [0, 0, -6, -12], mistakes={"le multiplicateur de L₂ doit annuler le coefficient de y dans L₃": [0, 2, 0, 4]})
wbm.record("CP2.4c", [1, 2, 2], mistakes={"la liste est [x, y, z], dans cet ordre": [2, 2, 1]})
wbm.record("CP2.4d", [[2, -2.5], [-1, 1.5]], decimals=2, mistakes={"la formule de l'inverse divise par le déterminant ad − bc": [[4, -5], [-2, 3]], "la formule échange a et d et change le signe de b et c, sans transposer": [[2, -1], [-2.5, 1.5]]})
wbm.record("CP2.4e", [-3, 2], mistakes={"la solution est B⁻¹ (1, 2), pas B (1, 2)": [13, 10]})
wbm.record("CP2.4f", 2, mistakes={"cherche une ligne qui est une combinaison des autres": 3, "le rang d'une matrice à 3 lignes est au plus 3": 4})
wbm.record("CP2.4g", 4)

WBM_ANSWER {"decimals": 0, "element_hashes": ["6c65e098e9d7b164aa4d66a213814e4f2525e0d7ff028855639cb13e5ed8aaa4", "03b789aef90f8c9009f6cc36c59622e3a3038b170355498e40a2050fc6383ff4", "42a2816a2737c0c9d9c966cb505447339633ecece6c86afa3d22f043a2dd99be", "ea8f3745a3cc2885c2a8745c9eb94f27e70ea349647dc25c02355d46b3d190e3"], "hash": "3dce7bc178f4082d83a9fa938a64df6ed30c113da9b82bc01733ecd5e56f8289", "id": "CP2.4a", "integer": true, "kind": "array", "mistakes": {"783c6e78d13b27ee020daa27c3a53631ec753b5e87677dc666b7c6974d41b00e": "l'opération porte aussi sur le second membre", "88ac1d83822d917f642892b9e619df469f5a1e9551c9bb178ef69f3db4518229": "L₃ ← L₃ + L₁ : on ajoute la première ligne"}, "shape": [4]}
📝 Ex CP2.4a : réponse enregistrée (array, 0 décimale).
WBM_ANSWER {"decimals": 0, "element_hashes": ["78cba0029ba41102bc256bcb5a7be7dbc7d7d0ed12ec2e7d8b9bf13576c40a36", "594656260b00b61ed9d99a6c882ab8a09ad835ab6cf343a8231a3fe95b49b523", "bc15119a29a55e4fc306403eee5aa3424d67e7be5d3f0f3ae27e685fb58

**CP2.5 — Les équations normales d'un petit modèle, avec et sans ridge**

In [7]:
wbm.record("CP2.5a", G_5)
wbm.record("CP2.5b", XY_5, mistakes={"XᵀY multiplie chaque colonne de X par y, exemple par exemple": [8, 13]})
wbm.record("CP2.5c", W_5, decimals=4, mistakes={"l'inverse de XᵀX divise par son déterminant": [24, 51], "la liste est [w₁, w₂]": W_5[::-1]})
wbm.record("CP2.5d", R_5, decimals=4, mistakes={"le résidu est y − Xw, pas Xw − y": [-r for r in R_5]})
wbm.record("CP2.5e", sum(r * r for r in R_5), decimals=4, mistakes={"la SCR additionne les carrés des résidus": sum(R_5), "la SCR est une somme, pas une moyenne": sum(r * r for r in R_5) / 4})
wbm.record("CP2.5f", WR_5, decimals=4, mistakes={"λI n'ajoute λ que sur la diagonale de XᵀX": solve2([[9, 6], [6, 9]], XY_5)})
wbm.record("CP2.5g", SCR_RIDGE_5, decimals=2, mistakes={"la SCR d'entraînement ne compte pas la pénalité λ ‖w‖²": SCR_RIDGE_5 + 3 * dot(WR_5, WR_5)})

WBM_ANSWER {"decimals": 0, "element_hashes": ["f32821ad816bd8dcce743971bb45b02d5c474d6520c9276225515d7462f940d2", "2bd262dbf394a445dc92f901cd026f839e47dae2e9113150d79bb3e7b027b612", "801119f1385267ee40ada55ceda7f6524a443415658527eadb2c994e04a0acfd", "d7f6c365af00215ef0f4630c2c0268d4b3693f69132ca26c705bdf95a1589696"], "hash": "b9f16db87aa127627c9b163a0253fd2db0882674a19c8a9d426d064d4adbc2e8", "id": "CP2.5a", "integer": true, "kind": "array", "shape": [2, 2]}
📝 Ex CP2.5a : réponse enregistrée (array, 0 décimale).
WBM_ANSWER {"decimals": 0, "element_hashes": ["5929cb01427bda5a64f02c73ba2334f048478b1be11fe76f63ebfc5ada94c0e5", "d6fffbd4e7cdd6d43b9b0123bc88045ca58cf36a745f6eefdd49847388adfe7b"], "hash": "af02b1d8e84a446f665dfd01fef10498670d7788242fb17a113de7e623049ff0", "id": "CP2.5b", "integer": true, "kind": "array", "mistakes": {"e320dbf345b6b4ef7ebfa7285b7266bdb2e4103346c337273bd58d9dcc825733": "XᵀY multiplie chaque colonne de X par y, exemple par exemple"}, "shape": [2]}
📝 Ex CP2.5b : 

**CP2.6 — Biais, variance et erreur quadratique d'un estimateur**

In [8]:
wbm.record("CP2.6a", laplace_6(0.2)[0], decimals=2, mistakes={"le biais est E[p̃] − p, dans cet ordre": -laplace_6(0.2)[0], "le biais est E[p̃] − p, pas E[p̃]": laplace_6(0.2)[0] + 0.2})
wbm.record("CP2.6b", laplace_6(0.2)[1], decimals=4, mistakes={"V(K + 1) = V(K) = n p (1 − p), et diviser par n + 2 divise la variance par (n + 2)²": 8 * 0.2 * 0.8 / 10, "V(K) = n p (1 − p), avec n = 8": 0.2 * 0.8 / 100})
wbm.record("CP2.6c", laplace_6(0.2)[2], decimals=4, mistakes={"l'erreur quadratique ajoute le carré du biais à la variance": laplace_6(0.2)[1], "le biais entre au carré": laplace_6(0.2)[1] + laplace_6(0.2)[0]})
wbm.record("CP2.6d", 0.2 * 0.8 / 8, decimals=4, mistakes={"la variance de K/n est p (1 − p) / n": 0.2 * 0.8})
wbm.record("CP2.6e", laplace_6(0.05)[2], decimals=4, mistakes={"l'erreur quadratique ajoute le carré du biais à la variance": laplace_6(0.05)[1], "le biais entre au carré": laplace_6(0.05)[1] + laplace_6(0.05)[0]})
wbm.record("CP2.6f", 1, choices=[1, 2], mistakes={"compare e) à l'erreur quadratique de p̂ pour p = 0,05": 2})

WBM_ANSWER {"decimals": 2, "hash": "22bc79ed60db713897ac857b607355436df3a3d02bae614bdaf5be6f5c002330", "hash_coarse": "823511255fd3d214d842d3398e66fd9516be7c7ffb70875677481e7eed1d7fb7", "id": "CP2.6a", "kind": "float", "magnitude": -2, "mistakes": {"9cb32789bd99acd037f9bbbcc253191d74a4a75e45c269d145eaf18fb0972e86": "le biais est E[p̃] − p, pas E[p̃]", "c4e5ac0a0ddb9697f1a57e83a6a19753e52e9928495000b3aa39dff463024b12": "le biais est E[p̃] − p, dans cet ordre"}, "sign": 1}
📝 Ex CP2.6a : réponse enregistrée (float, 2 décimales).
WBM_ANSWER {"coarse_any": {"2": ["76a5aa54e7eb23158f6353edb9c6d77ff2ae29e71b68074c6110dba7e7d617e3"]}, "decimals": 4, "hash": "43a4b5179f378f521cb7954a671b0353ac1c96f8c23be0d87bfcfb384f013468", "hash_coarse": "01028fd70118c49cf2d388c05703d4d2693eae8c31c5b7ad4fe22fff1e9cc9c2", "id": "CP2.6b", "kind": "float", "magnitude": -2, "mistakes": {"561548da36a89efaf7c07e39bc6311bf18ddac9d1dfc2dc0dcb734d0d392d0f0": "V(K) = n p (1 − p), avec n = 8", "85fb3ac61ad19e5e787e60e42

**CP2.7 — Convexité, enveloppe convexe et marge**

In [9]:
wbm.record("CP2.7a", [[0, 0], [4, 0], [5, 3], [2, 5]], mistakes={"le sens direct tourne vers la gauche, comme le sens contraire des aiguilles": [[0, 0], [2, 5], [5, 3], [4, 0]], "un point intérieur n'est pas un sommet : teste (1, 2) avec un produit vectoriel": [[0, 0], [4, 0], [5, 3], [2, 5], [1, 2]]})
wbm.record("CP2.7b", 15.5, decimals=2, mistakes={"l'aire d'un triangle est la moitié de la valeur absolue du produit vectoriel": 31})
wbm.record("CP2.7c", False, mistakes={"teste (1, 3) contre chaque côté avec un produit vectoriel, sens direct : il doit être à gauche de chacun": True})
wbm.record("CP2.7d", margin_7((1, 1), -6.5), decimals=3, mistakes={"la marge géométrique divise par ‖w‖": 1.5, "‖w‖ est la racine de 1² + 1²": 0.75})
wbm.record("CP2.7e", margin_7((2, 1), -9.5), decimals=3, mistakes={"la marge maximale est la moitié de la distance entre les deux enveloppes": 2 * margin_7((2, 1), -9.5)})
wbm.record("CP2.7f", [0.8, 0.4, -3.8], decimals=2, mistakes={"forme canonique : divise (w, b) par le plus petit yᵢ (w · xᵢ + b)": [2, 1, -9.5], "la classe +1 doit être du côté où w · x + b > 0": [-0.8, -0.4, 3.8]})

WBM_ANSWER {"decimals": 0, "element_hashes": ["9fbce5edd67bd5013b9acb5dc176c6ce0455499aeee32d7e847c138247c95395", "9bcb311cb3db7b8c73d5b1c1bf85d53492a3b741a6ee1a84c6128432ceb5eb38", "accbef35921a9df127aacf6b727148368bc8aa04ca6c3e611705e5ef12a21a05", "345b07c3a07e2ca6efb74390f39677112eb4babac92969af6ee48c8493195a74", "8f3d1710df2f9af79db9986bca75dac25f5508efad460efb57a43be93a790ef2", "9dfe54abd72450bbb90c5249bbfbc72c99d030536f495c0d7a4a2e805833dee7", "f80cf933ee97c0416c9c67fd35dbe087c597a911ed4dfcb427a58fcb32722713", "d693164805d15e4b10a2e2efbbd5e3c6891ed9d78435e6f9a74639ab95f7cc21"], "hash": "1256862269e5b5ff746cff7f0ea173f7114486bbed5dc1c7dcfd35c781127f41", "id": "CP2.7a", "integer": true, "kind": "array", "mistakes": {"21c0d695c75eeb1147d7d0c26fa70f57e9b184e16b099dbc6d113985ee5c593b": "un point intérieur n'est pas un sommet : teste (1, 2) avec un produit vectoriel", "8ef5e363cef628ecf962976bd2b32fd178458cac314cb3fe05b99b7a15cd5d88": "le sens direct tourne vers la gauche, comme le sen

**CP2.8 — Markov, Tchebychev et Hoeffding**

In [10]:
wbm.record("CP2.8a", 120 / 300, decimals=2, mistakes={"Markov divise la moyenne par le seuil": 300 / 120})
wbm.record("CP2.8b", 30 ** 2 / (300 - 120) ** 2, decimals=4, mistakes={"Tchebychev divise la variance par le carré de l'écart": 30 / 180, "l'écart à la moyenne est 300 − 120": 900 / 300 ** 2})
wbm.record("CP2.8c", 720, mistakes={"Tchebychev utilise la variance σ² = 900": 120, "l'écart entre au carré : t² = 25": 3600})
wbm.record("CP2.8d", math.sqrt(math.log(2 / 0.02) / (2 * 1200)), decimals=4, mistakes={"le rayon bilatéral utilise ln(2/δ)": math.sqrt(math.log(1 / 0.02) / 2400), "ln(2/δ) se divise par 2n": math.sqrt(math.log(100) / 1200)})
wbm.record("CP2.8e", 2 * math.exp(-2 * 1200 * 0.05 ** 2), decimals=4, mistakes={"la borne bilatérale a un facteur 2": math.exp(-6)})
wbm.record("CP2.8f", math.ceil(math.log(2 / 0.02) / (2 * 0.025 ** 2)), mistakes={"n doit être au moins égal à la formule : arrondis au-dessus": 3684, "le rayon bilatéral utilise ln(2/δ)": 3130})

WBM_ANSWER {"decimals": 2, "hash": "b19347423b49b1032dded5881656144ca6ece1785e1dc46494aa30c81b1354fc", "hash_coarse": "d991b2e1b67c406532cd95cc08bcf5a9023f6126458dd227d57b256e6181b380", "id": "CP2.8a", "kind": "float", "magnitude": -1, "mistakes": {"5e72f78455beb87d259aa1b530517021b15a22062d3c87deb21eb52e2c13039a": "Markov divise la moyenne par le seuil"}, "sign": 1}
📝 Ex CP2.8a : réponse enregistrée (float, 2 décimales).
WBM_ANSWER {"coarse_any": {"2": ["6b056e6431482ae68eb77b5c10623134d126a26ef25149ca26d53695f1cb0e1b"]}, "decimals": 4, "hash": "02384333d22a11e9ebf7d3a08deaf2a60adf46fa4a21a22c3a4def435d199178", "hash_coarse": "0771c534c863edbaabfdd1b821f5472736e4934b32712f82c33f632d7add5065", "id": "CP2.8b", "kind": "float", "magnitude": -2, "mistakes": {"9f91b7d24aa956c0510535723f7af78db79374e9ec24e6c6226061d4eb9c9e11": "Tchebychev divise la variance par le carré de l'écart", "af3d9edb5cc631955dcefbc1c38265f32815bb9d095a686e53debc156da7c2f0": "l'écart à la moyenne est 300 − 120"}, "s

**CP2.10 — Corriger une copie : cinq erreurs (les lettres seulement)**

In [11]:
wbm.record("CP2.10a", "BCEGH", mistakes={"il manque une des lignes fausses : il y en a exactement cinq": {'BCEG', 'BCEH', 'BCGH', 'BEGH', 'CEGH'}, "une des lignes citées est juste : relis-les une par une, avec la règle qu'elle utilise": {'ABCEGH', 'BCDEGH', 'BCEFGH'}, "une ligne juste est citée à la place d'une ligne fausse": {'ABCEG', 'ABCEH', 'ABCGH', 'ABEGH', 'ACEGH', 'BCDEG', 'BCDEH', 'BCDGH', 'BCEFG', 'BCEFH', 'BCFGH', 'BDEGH', 'BEFGH', 'CDEGH', 'CEFGH'}, "ce sont les trois lignes justes ; la question demande les cinq lignes fausses": 'ADF'})

WBM_ANSWER {"hash": "8171734e2894d8c242ae89e7f43fbb4b1b8ed9f9732e613cbc51481987a176d3", "id": "CP2.10a", "kind": "str", "mistakes": {"1017fe63c7a865985357ad71b1b149dea46de4d8cc5b6df8d447e792abe7ad19": "une des lignes citées est juste : relis-les une par une, avec la règle qu'elle utilise", "127dd4bc62ca979a8e636fd8a933ac6ec46769c69f9533e6238e1343fb813f6b": "une ligne juste est citée à la place d'une ligne fausse", "26e584f677827e0512b074114ae1c07c3a3f46ec9ec25be1759d569746aadf09": "une des lignes citées est juste : relis-les une par une, avec la règle qu'elle utilise", "3e96e38e75b7b0f74376b07f0ab78cc21a131942bc7a56f5cd640a60a2ebb006": "il manque une des lignes fausses : il y en a exactement cinq", "4fa8424a1c08413db122b0e11e4d30d6de16f85182669d15df69cf0d1196ed63": "une ligne juste est citée à la place d'une ligne fausse", "54e20ceb1a66a6aa65f28d1f9589751a81b3236b470b249f6b5b0ab74d30eab5": "une ligne juste est citée à la place d'une ligne fausse", "62b95b5ff5d1b2ec6249a908a3d9e555336aa

**CP2.11 — Problème de synthèse : ajuster un modèle, mesurer son erreur, décider**

In [12]:
wbm.record("CP2.11a", [G_11[1][1], XY_11[1]], mistakes={"Σxᵢ² additionne les carrés des tailles, et Σxᵢyᵢ les produits nuit par nuit": [30 ** 2, 30 * 90]})
wbm.record("CP2.11b", [A_11, B_11], decimals=4, mistakes={"la liste est [a, b] : l'ordonnée à l'origine d'abord": [B_11, A_11]})
wbm.record("CP2.11c", A_11 + 9 * B_11, decimals=2, mistakes={"la prédiction est a + b x, avec a l'ordonnée à l'origine": B_11 + 9 * A_11})
wbm.record("CP2.11d", SCR_11, decimals=3, mistakes={"la SCR mesure les résidus du modèle, pas les écarts à la moyenne de y": SCT_11})
wbm.record("CP2.11e", 1 - SCR_11 / SCT_11, decimals=4, mistakes={"R² = 1 − SCR / SCT": SCR_11 / SCT_11})
wbm.record("CP2.11f", aic_11(SCR_11, 2), decimals=1, mistakes={"l'AIC prend le logarithme de SCR / n": N_11 * math.log(SCR_11) + 4, "ln est le logarithme népérien": N_11 * math.log10(SCR_11 / N_11) + 4})
wbm.record("CP2.11g", 1, choices=[0, 1, 2], mistakes={"un coefficient de plus coûte 2 dans l'AIC : la SCR baisse-t-elle assez ?": 2, "compare les trois AIC : le plus petit est retenu": 0})
wbm.record("CP2.11h", LOW_11, decimals=4, mistakes={"une borne d'un seul côté utilise ln(1/δ), pas ln(2/δ)": 344 / 400 - math.sqrt(math.log(2 / 0.05) / 800), "ln(1/δ) se divise par 2n": 344 / 400 - math.sqrt(math.log(20) / 400)})
wbm.record("CP2.11i", False, mistakes={"compare h) à 0,80": True})

WBM_ANSWER {"decimals": 0, "element_hashes": ["c16b0221ad6fcb15809d92b463b3abb6b2d1d6976f871c166311c48e4592461a", "f2772d46d74101471fe47c18550cb379b709bc476a2ed7b89f59d0359f7b00c4"], "hash": "68d8c62df9f037c473559cb7ff7871096324e78271df69b6d6f4e1d3a507871c", "id": "CP2.11a", "integer": true, "kind": "array", "mistakes": {"f37e5d7e646bc2a67ae534c69bdfe343365fc709e38e08072fbfed0fe04dad19": "Σxᵢ² additionne les carrés des tailles, et Σxᵢyᵢ les produits nuit par nuit"}, "shape": [2]}
📝 Ex CP2.11a : réponse enregistrée (array, 0 décimale).
WBM_ANSWER {"decimals": 4, "element_coarse": [["cf40019fc8a8b7fbd658e3198f32f85809bb6f541766d20744568f9834179fb1"], ["3bc2c1005ec6959c8ceb967eb18c24287afc3daabef547fbe6b86992046a3859"]], "element_coarse_any": [["49dde807acb2ad2d62c80e83dfa9f7f26990c7efffb61c1679fabe07e129d140", "b9ea7e4b2d0dc69eb1cd88e8e01819bf6f36c387e76277a3420945a1e90caa78", "aecfffca0e159e8a202aaa41c9bb4d980d64165dac4f00d51073e42ea1774a47"], ["e62e7f2b3ce1b19228e9ac0d976747921bc572f2f